# siebel_id_test_databricks

Case-driven tests for incremental output maintenance using test baselines persisted in the `shebang` tables.

Rules:
- each case has one markdown cell and one code cell
- each case calls `run_incremental_update_databricks(...)` from `siebel_id_update_databricks.ipynb`
- each case starts by restoring `shebang.direct_bank`, `shebang.ggm_np`, and `shebang.output` from the `*_test` tables
- rerun the initialization cell to refresh the `*_test` tables from `use_cases/siebel_id/test/*.csv`
- this notebook currently documents 15 replayable cases (including historical direct_bank connectivity, active-seed eligibility checks, cross-source rel_id bridge connectivity, multi-seed deterministic winner selection, and diagnostic-only duplicate active seed-key validation)

Databricks assumptions:
- an active cluster is attached
- `spark` session is available
- project is opened from a Databricks Repo path

In [ ]:
# Databricks runtime parameters (edit these for your workspace)
CATALOG = None  # e.g. 'main' in Unity Catalog; keep None for hive_metastore/default
WORK_SCHEMA = 'shebang'
STATE_SCHEMA = 'shebang_state'

if CATALOG:
    WORK_DB = f"{CATALOG}.{WORK_SCHEMA}"
    STATE_DB = f"{CATALOG}.{STATE_SCHEMA}"
else:
    WORK_DB = WORK_SCHEMA
    STATE_DB = STATE_SCHEMA

print('Configured WORK_DB:', WORK_DB)
print('Configured STATE_DB:', STATE_DB)

In [ ]:
from pathlib import Path
import json
import re
from pyspark.sql import SparkSession, functions as F

assert isinstance(spark, SparkSession), "This notebook must run with an active Databricks Spark session."

ROOT_DIR = Path.cwd().resolve()
while ROOT_DIR != ROOT_DIR.parent and not (ROOT_DIR / 'notebooks' / 'siebel_id_update_databricks.ipynb').exists():
    ROOT_DIR = ROOT_DIR.parent

if not (ROOT_DIR / 'notebooks' / 'siebel_id_update_databricks.ipynb').exists():
    raise RuntimeError(f'Could not resolve repository root from {Path.cwd()}')

# Load siebel_id_update_databricks code cells without nbformat dependency.
update_nb_path = ROOT_DIR / 'notebooks' / 'siebel_id_update_databricks.ipynb'
with open(update_nb_path, 'r', encoding='utf-8') as f:
    update_nb = json.load(f)

for cell in update_nb.get('cells', []):
    if cell.get('cell_type') != 'code':
        continue
    code = '\n'.join(cell.get('source', []))
    # Import only library-like definitions; skip runnable/demo cells.
    if 'stats = run_incremental_update_databricks(' in code or 'stats = run_incremental_update(' in code:
        continue
    exec(code, globals())

if 'run_incremental_update_databricks' not in globals():
    raise RuntimeError('Failed to load run_incremental_update_databricks from siebel_id_update_databricks.ipynb')

checkpoint_dir = '/tmp/siebel_id_test_databricks_ckpt'
try:
    spark.sparkContext.setCheckpointDir(checkpoint_dir)
except Exception:
    pass

# Allow override from the Databricks runtime-parameters cell.
WORK_DB = globals().get('WORK_DB', 'shebang')
STATE_DB = globals().get('STATE_DB', 'shebang_state')
TEST_INPUT_DIR = ROOT_DIR / 'use_cases' / 'siebel_id' / 'test'
WORK_TABLE_NAMES = ['direct_bank', 'ggm_np', 'output']


def table_exists(full_table_name: str) -> bool:
    return spark.catalog.tableExists(full_table_name)


def to_table_name(path_obj: Path) -> str:
    return re.sub(r'[^a-z0-9_]', '_', path_obj.stem.lower())


def write_dataframe_to_table(full_table_name: str, df) -> None:
    if table_exists(full_table_name):
        overwrite_table_in_place(spark, full_table_name, df, fmt='delta')
        return

    try:
        df.write.mode('overwrite').format('delta').saveAsTable(full_table_name)
    except Exception as exc:
        if 'LOCATION_ALREADY_EXISTS' not in str(exc):
            raise
        spark.sql(f'DROP TABLE IF EXISTS {full_table_name}')
        df.write.mode('overwrite').format('delta').saveAsTable(full_table_name)


def ensure_test_tables_exist() -> None:
    spark.sql(f'CREATE DATABASE IF NOT EXISTS {WORK_DB}')

    loaded_table_names = set()
    for csv_path in sorted(TEST_INPUT_DIR.glob('*.csv')):
        base_table_name = to_table_name(csv_path)
        full_table_name = f'{WORK_DB}.{base_table_name}_test'
        df = (
            spark.read.option('header', True)
            .option('sep', '\t')
            .option('inferSchema', True)
            .csv(str(csv_path))
        )
        write_dataframe_to_table(full_table_name, df)
        loaded_table_names.add(base_table_name)

    missing_tables = sorted(set(WORK_TABLE_NAMES) - loaded_table_names)
    if missing_tables:
        raise FileNotFoundError(
            f'Missing test CSVs for required tables: {missing_tables} in {TEST_INPUT_DIR}'
        )


def reset_sequence_to_test_tables() -> None:
    ensure_test_tables_exist()

    for table_name in WORK_TABLE_NAMES:
        source_table = f'{WORK_DB}.{table_name}_test'
        target_table = f'{WORK_DB}.{table_name}'
        write_dataframe_to_table(target_table, spark.table(source_table))

    spark.sql(f'DROP TABLE IF EXISTS {STATE_DB}.edge_snapshot')
    spark.sql(f'DROP TABLE IF EXISTS {STATE_DB}.seed_snapshot')
    spark.sql(f'DROP TABLE IF EXISTS {STATE_DB}.seed_membership')
    spark.sql(f'DROP TABLE IF EXISTS {STATE_DB}.run_audit')
    spark.catalog.clearCache()


def pair_set(df):
    return {(str(r['REL_ID']), str(r['REL_ID_REGIE_KLANT'])) for r in df.select('REL_ID', 'REL_ID_REGIE_KLANT').collect()}


def run_once():
    return run_incremental_update_databricks(spark, source_db=WORK_DB, target_db=WORK_DB, state_db=STATE_DB, show_samples=False)


def prepare_case_baseline() -> set:
    reset_sequence_to_test_tables()
    run_once()
    return pair_set(spark.table(f"{WORK_DB}.output"))


def assert_delta(case_name: str, prev_pairs: set, cur_pairs: set, expected_inserted: list, expected_removed: list) -> None:
    inserted = sorted(cur_pairs - prev_pairs)
    removed = sorted(prev_pairs - cur_pairs)
    exp_inserted = sorted(expected_inserted)
    exp_removed = sorted(expected_removed)

    if inserted != exp_inserted or removed != exp_removed:
        raise AssertionError(
            f"{case_name} FAIL\nExpected inserted={exp_inserted} removed={exp_removed}\nActual inserted={inserted} removed={removed}"
        )

    print(f"{case_name}: PASS")


def assert_output_has_no_duplicates(case_name: str) -> None:
    output_df = spark.table(f"{WORK_DB}.output")
    total_rows = output_df.count()
    distinct_rows = output_df.dropDuplicates(['REL_ID', 'REL_ID_REGIE_KLANT']).count()

    if total_rows != distinct_rows:
        raise AssertionError(
            f"{case_name} FAIL\nOutput contains duplicate rows: total_rows={total_rows} distinct_rows={distinct_rows}"
        )

    rel_multi_seed = (
        output_df.groupBy('REL_ID')
        .agg(F.countDistinct('REL_ID_REGIE_KLANT').alias('seed_count'))
        .filter(F.col('seed_count') > 1)
    )
    if rel_multi_seed.count() > 0:
        raise AssertionError(f"{case_name} FAIL\nN->1 violated: some REL_ID map to more than one REL_ID_REGIE_KLANT")

    active_seed_ids = (
        spark.table(f"{WORK_DB}.direct_bank")
        .filter(
            (F.col('drc_bnk_f') == F.lit('Y'))
            & (F.to_timestamp(F.col('edl_valid_to_dts')) == F.to_timestamp(F.lit('9999-12-31 00:00:00')))
        )
        .select(F.col('rel_id').cast('string').alias('REL_ID_REGIE_KLANT'))
        .dropDuplicates(['REL_ID_REGIE_KLANT'])
    )

    invalid_output = output_df.select('REL_ID_REGIE_KLANT').dropDuplicates(['REL_ID_REGIE_KLANT']).join(
        active_seed_ids, 'REL_ID_REGIE_KLANT', 'left_anti'
    )
    if invalid_output.count() > 0:
        raise AssertionError(f"{case_name} FAIL\nOutput contains REL_ID_REGIE_KLANT values that are not active direct_bank seeds")

    print(f"{case_name}: output invariants OK (unique + N->1 + active seed eligibility)")


def to_display_pandas(df):
    preview_df = df
    for field in preview_df.schema.fields:
        if field.dataType.typeName() in {'timestamp', 'date'}:
            preview_df = preview_df.withColumn(field.name, F.col(field.name).cast('string'))
    return preview_df.toPandas()


def show_dataframe(title: str, df, order_by: list[str] | None = None) -> None:
    print(title)
    preview_df = df.orderBy(*order_by) if order_by else df
    display(to_display_pandas(preview_df))


def show_case_tables(case_name: str) -> None:
    direct_df = spark.table(f"{WORK_DB}.direct_bank")
    ggm_df = spark.table(f"{WORK_DB}.ggm_np")
    output_df = spark.table(f"{WORK_DB}.output")

    print(f"{case_name} input/output preview")
    show_dataframe('direct_bank', direct_df, order_by=['rel_id', 'edl_valid_from_dts'])
    show_dataframe('ggm_np', ggm_df, order_by=['rel_id', 'edl_valid_from_dts'])
    show_dataframe('output', output_df, order_by=['REL_ID'])


def update_column_with_fallback(full_table: str, col_name: str, condition, new_value) -> None:
    # SQL UPDATE may fail depending on provider capabilities or permissions; fallback preserves behavior.
    def _transform(df):
        return df.withColumn(col_name, F.when(condition, new_value).otherwise(F.col(col_name)))

    mutate_table_with_in_place_overwrite(spark, full_table, _transform)


print('Databricks test harness initialized for shebang test baselines')
print('Using WORK_DB:', WORK_DB)
print('Using STATE_DB:', STATE_DB)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/24 11:53:21 WARN Utils: Your hostname, PM2745.local, resolves to a loopback address: 127.0.0.1; using 10.7.1.131 instead (on interface en0)
26/06/24 11:53:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 11:53:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Test harness initialized for shebang test baselines


In [ ]:
# Recover SparkSession if previous JVM backend died between runs.
from pyspark.sql import SparkSession

try:
    spark.range(1).count()
    print('Spark session is alive.')
except Exception:
    # Reattach to active Databricks session.
    SparkSession._instantiatedSession = None
    SparkSession._activeSession = None
    spark = SparkSession.builder.getOrCreate()
    try:
        spark.sparkContext.setCheckpointDir(checkpoint_dir)
    except Exception:
        pass
    print('Spark session was reattached.')

Spark session is alive.


In [3]:
# Hard reset helper for true scratch replay
spark.sql(f"DROP DATABASE IF EXISTS {STATE_DB} CASCADE")
spark.sql(f"DROP DATABASE IF EXISTS {WORK_DB} CASCADE")
spark.catalog.clearCache()
print("Hard reset complete: dropped shebang and shebang_state, including test tables.")

26/06/24 11:53:26 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/06/24 11:53:26 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore jguerrrero@127.0.0.1


Hard reset complete: dropped shebang and shebang_state, including test tables.


26/06/24 11:53:27 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist
26/06/24 11:53:27 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist


In [4]:
reset_sequence_to_test_tables()
baseline_stats = run_once()
show_case_tables('Baseline after initialization')
print('Baseline stats:', baseline_stats)

26/06/24 11:53:27 WARN ObjectStore: Failed to get database shebang, returning NoSuchObjectException
26/06/24 11:53:27 WARN ObjectStore: Failed to get database shebang, returning NoSuchObjectException
26/06/24 11:53:27 WARN ObjectStore: Failed to get database shebang, returning NoSuchObjectException
26/06/24 11:53:28 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/06/24 11:53:28 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/06/24 11:53:29 WARN ObjectStore: Failed to get database shebang_state, returning NoSuchObjectException
26/06/24 11:53:29 WARN ObjectStore: Failed to get database shebang_state, returning NoSuchObjectException
26/06/24 11:53:29 WARN ObjectStore: Failed to get database shebang_state, returning NoSuchObjectException
26/06/24 11:53:29 WARN ObjectStore: Failed to get database shebang_state, returning NoSuchObjectException
2

Baseline after initialization input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
107,118492640,2024-02-16 23:34:22,2024-04-24 16:33:50,1-1TWAZ5GR,N
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640


Baseline stats: {'changed_edges': 14, 'added_seeds': 2, 'removed_seeds': 0, 'impacted_rel_ids': 6, 'output_rows': 6}


## Sequence Initialization

Run this cell before the cases to refresh the `*_test` tables from `use_cases/siebel_id/test/*.csv`, restore the working `shebang` tables from those test tables, and synchronize incremental state.

## Case 1: new direct_bank active seed

Scenario:
- insert a new `direct_bank` row where `drc_bnk_f='Y'` and `edl_valid_to_dts='9999-12-31'`

Expected delta/output:
- inserted: `('130000001', '130000001')`
- removed: none

In [5]:
prev_pairs = prepare_case_baseline()

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (130000001, to_timestamp('2026-05-18 09:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-Z-NEW1', 'Y')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 1')
assert_delta('Case 1', prev_pairs, cur_pairs, [('130000001', '130000001')], [])
assert_output_has_no_duplicates('Case 1')

Case 1 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y
9,130000001,2026-05-18 09:00:00,9999-12-31 00:00:00,1-Z-NEW1,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
107,118492640,2024-02-16 23:34:22,2024-04-24 16:33:50,1-1TWAZ5GR,N
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640
6,130000001,130000001


Case 1: PASS
Case 1: output invariants OK (unique + N->1 + active seed eligibility)


## Case 2: flip seed off with fallback mutation

Scenario:
- change `drc_bnk_f` from `Y` to `N` for seed `rel_id=118492640`
- SQL UPDATE may be unsupported or restricted, so fallback uses same-table DataFrame overwrite

Expected delta/output:
- inserted: none
- removed: `('11410074', '118492640')`, `('117786942', '118492640')`, `('118492640', '118492640')`

In [6]:
prev_pairs = prepare_case_baseline()

cond = (
    (F.col('rel_id') == F.lit(118492640))
    & (F.col('np_sbl_id') == F.lit('1-1TWAZ5GR'))
    & (F.col('drc_bnk_f') == F.lit('Y'))
    & (F.col('edl_valid_to_dts') == F.to_timestamp(F.lit('9999-12-31 00:00:00')))
 )
update_column_with_fallback(f"{WORK_DB}.direct_bank", 'drc_bnk_f', cond, F.lit('N'))

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 2')
assert_delta(
    'Case 2',
    prev_pairs,
    cur_pairs,
    [],
    [('11410074', '118492640'), ('117786942', '118492640'), ('118492640', '118492640')],
)
assert_output_has_no_duplicates('Case 2')

Case 2 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,N


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
107,118492640,2024-02-16 23:34:22,2024-04-24 16:33:50,1-1TWAZ5GR,N
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118220032,115379534
2,118220032,118220032


Case 2: PASS
Case 2: output invariants OK (unique + N->1 + active seed eligibility)


## Case 3: ggm_np rel_id reassignment via fallback

Scenario:
- reassign `ggm_np.rel_id` from `130000002` to `130000003` for `ikb_no='1-1N09820A'`

Expected delta/output:
- inserted: `('130000003', '118220032')`
- removed: `('130000002', '118220032')`

In [7]:
prev_pairs = prepare_case_baseline()
spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (130000002, to_timestamp('2026-05-18 11:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-1N09820A', 'N')
    """
)
_ = run_once()  # absorb inserted row first
prev_pairs = pair_set(spark.table(f"{WORK_DB}.output"))

cond = (
    (F.col('rel_id') == F.lit(130000002))
    & (F.col('ikb_no') == F.lit('1-1N09820A'))
    & (F.col('edl_valid_from_dts') == F.to_timestamp(F.lit('2026-05-18 11:00:00')))
 )
update_column_with_fallback(f"{WORK_DB}.ggm_np", 'rel_id', cond, F.lit(130000003))

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 3')
assert_delta('Case 3', prev_pairs, cur_pairs, [('130000003', '118220032')], [('130000002', '118220032')])
assert_output_has_no_duplicates('Case 3')

Case 3 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N
111,118492640,2025-10-01 07:27:12.034435,9999-12-31 00:00:00,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640
6,118220032,130000003


Case 3: PASS
Case 3: output invariants OK (unique + N->1 + active seed eligibility)


## Case 4: idempotent no-change run

Scenario:
- execute incremental update twice with no source mutations

Expected delta/output:
- inserted: none
- removed: none

In [ ]:
prev_pairs = prepare_case_baseline()
audit_table = f"{STATE_DB}.run_audit"
baseline_audit_rows = spark.table(audit_table).count()

_ = run_once()  # no mutation
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 4')
assert_delta('Case 4', prev_pairs, cur_pairs, [], [])
assert_output_has_no_duplicates('Case 4')

current_audit_rows = spark.table(audit_table).count()
if current_audit_rows != baseline_audit_rows + 1:
    raise AssertionError(
        f"Case 4 FAIL\nrun_audit row growth mismatch: baseline={baseline_audit_rows} current={current_audit_rows}"
    )

print('Case 4: run_audit append behavior OK')

Case 4 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
107,118492640,2024-02-16 23:34:22,2024-04-24 16:33:50,1-1TWAZ5GR,N
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640


Case 4: PASS
Case 4: output invariants OK (unique + N->1 + active seed eligibility)


## Case 5: new ikb_no for the same rel_id

Scenario:
- insert a new `ggm_np` row with a fresh `ikb_no` for existing `rel_id=118220032`

Expected delta/output:
- inserted: none
- removed: none
- the output row for `118220032` remains unique and unchanged

In [9]:
prev_pairs = prepare_case_baseline()

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (118220032, to_timestamp('2026-05-18 12:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-ALT-IKB-118220032', 'N')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 5')
assert_delta('Case 5', prev_pairs, cur_pairs, [], [])
assert_output_has_no_duplicates('Case 5')

Case 5 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
108,118492640,2024-02-16 23:34:22,2024-04-24 16:33:50,1-1TWAZ5GR,N
109,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
110,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
111,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640


Case 5: PASS
Case 5: output invariants OK (unique + N->1 + active seed eligibility)


## Case 6: duplicate source row does not duplicate output

Scenario:
- duplicate an existing active `ggm_np` row in the input

Expected delta/output:
- inserted: none
- removed: none
- output remains unique by `(REL_ID, REL_ID_REGIE_KLANT)`

In [10]:
prev_pairs = prepare_case_baseline()

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np
    SELECT rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f
    FROM {WORK_DB}.ggm_np
    WHERE rel_id = 118220032
      AND ikb_no = '1-1N09820A'
      AND edl_valid_to_dts = to_timestamp('9999-12-31 00:00:00')
    LIMIT 1
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 6')
assert_delta('Case 6', prev_pairs, cur_pairs, [], [])
assert_output_has_no_duplicates('Case 6')

26/06/24 11:54:47 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


Case 6 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
108,118492640,2024-02-16 23:34:22,2024-04-24 16:33:50,1-1TWAZ5GR,N
109,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
110,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
111,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640


Case 6: PASS
Case 6: output invariants OK (unique + N->1 + active seed eligibility)


## Case 7: new rel_id for an existing ikb_no

Scenario:
- insert a new `ggm_np.rel_id` that reuses existing `ikb_no='1-1N09820A'`

Expected delta/output:
- inserted: `('130000004', '118220032')`
- removed: none

In [11]:
prev_pairs = prepare_case_baseline()

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (130000004, to_timestamp('2026-05-18 12:30:00'), to_timestamp('9999-12-31 00:00:00'), '1-1N09820A', 'N')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 7')
assert_delta('Case 7', prev_pairs, cur_pairs, [('130000004', '118220032')], [])
assert_output_has_no_duplicates('Case 7')

Case 7 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N
111,118492640,2025-10-01 07:27:12.034435,9999-12-31 00:00:00,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640
6,118220032,130000004


Case 7: PASS
Case 7: output invariants OK (unique + N->1 + active seed eligibility)


## Case 8: simultaneous new relationships in both tables

Scenario:
- insert a new active `direct_bank` seed and a new `ggm_np` row at the same time
- both rows share the same business identifier, so the new `ggm_np` relation should inherit the new seed

Expected delta/output:
- inserted: `('130000010', '130000010')`, `('130000011', '130000010')`
- removed: none

In [12]:
prev_pairs = prepare_case_baseline()

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (130000010, to_timestamp('2026-05-18 13:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-CROSS-LINK-01', 'Y')
    """
)
spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (130000011, to_timestamp('2026-05-18 13:05:00'), to_timestamp('9999-12-31 00:00:00'), '1-CROSS-LINK-01', 'N')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 8')
assert_delta('Case 8', prev_pairs, cur_pairs, [('130000010', '130000010'), ('130000011', '130000010')], [])
assert_output_has_no_duplicates('Case 8')

Case 8 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y
9,130000010,2026-05-18 13:00:00,9999-12-31 00:00:00,1-CROSS-LINK-01,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N
111,118492640,2025-10-01 07:27:12.034435,9999-12-31 00:00:00,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640
6,130000010,130000010
7,130000010,130000011


Case 8: PASS
Case 8: output invariants OK (unique + N->1 + active seed eligibility)


## Case 9: simultaneous cross-table new rel_ids to existing active seed

Scenario:
- Insert new direct_bank and ggm_np rows in the same run.
- Both rows introduce new rel_id values, but connect through existing active identifier `1-1N09820A`.
- No new seed is introduced (`drc_bnk_f = 'N'` on direct_bank row).

Expected delta/output after first run:
- inserted: `('130000012', '118220032')`, `('130000013', '118220032')`
- removed: none

Expected delta/output after second run (idempotency):
- inserted: none
- removed: none

In [13]:
prev_pairs = prepare_case_baseline()

# direct_bank new rel_id linked to existing network, but not a new seed
spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (130000012, to_timestamp('2026-05-18 13:20:00'), to_timestamp('9999-12-31 00:00:00'), '1-1N09820A', 'N')
    """
)

# ggm_np new rel_id linked to same existing identifier
spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (130000013, to_timestamp('2026-05-18 13:21:00'), to_timestamp('9999-12-31 00:00:00'), '1-1N09820A', 'N')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 9')
assert_delta(
    'Case 9',
    prev_pairs,
    cur_pairs,
    [('130000012', '118220032'), ('130000013', '118220032')],
    [],
)
assert_output_has_no_duplicates('Case 9')

# Optional idempotency proof: immediate rerun without new mutations
prev_pairs_2 = cur_pairs
_ = run_once()
cur_pairs_2 = pair_set(spark.table(f"{WORK_DB}.output"))
assert_delta('Case 9 idempotency', prev_pairs_2, cur_pairs_2, [], [])
assert_output_has_no_duplicates('Case 9 idempotency')

Case 9 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y
9,130000012,2026-05-18 13:20:00,9999-12-31 00:00:00,1-1N09820A,N


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N
111,118492640,2025-10-01 07:27:12.034435,9999-12-31 00:00:00,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640
6,118220032,130000012
7,118220032,130000013


Case 9: PASS
Case 9: output invariants OK (unique + N->1 + active seed eligibility)
Case 9 idempotency: PASS
Case 9 idempotency: output invariants OK (unique + N->1 + active seed eligibility)


## Case 10: historical direct_bank rel_id is included in output REL_ID

Scenario:
- Insert a historical direct_bank row (closed validity window).
- The row links a new rel_id to existing identifier `1-1N09820A`.
- Because direct_bank historical edges are part of the graph, the new rel_id must appear in output.

Expected delta/output:
- inserted: `('130000100', '118220032')`
- removed: none

In [14]:
prev_pairs = prepare_case_baseline()

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (130000100, to_timestamp('2026-06-10 09:00:00'), to_timestamp('2026-06-10 09:05:00'), '1-1N09820A', 'N')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 10')
assert_delta('Case 10', prev_pairs, cur_pairs, [('130000100', '118220032')], [])
assert_output_has_no_duplicates('Case 10')

Case 10 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y
9,130000100,2026-06-10 09:00:00,2026-06-10 09:05:00,1-1N09820A,N


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
107,118492640,2024-02-16 23:34:22,2024-04-24 16:33:50,1-1TWAZ5GR,N
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640
6,118220032,130000100


Case 10: PASS
Case 10: output invariants OK (unique + N->1 + active seed eligibility)


## Case 11: closed `drc_bnk_f='Y'` row is not eligible as REL_ID_REGIE_KLANT

Scenario:
- Insert a direct_bank row with `drc_bnk_f='Y'` but closed `edl_valid_to_dts`.
- The new rel_id is connected to existing identifier `1-1N09820A`.
- The row should contribute REL_ID connectivity, but not become an active seed.

Expected delta/output:
- inserted: `('110000000', '118220032')`
- removed: none
- additional guard: no row with `REL_ID_REGIE_KLANT='110000000'`

In [15]:
prev_pairs = prepare_case_baseline()

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (110000000, to_timestamp('2026-06-10 10:00:00'), to_timestamp('2026-06-10 10:05:00'), '1-1N09820A', 'Y')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 11')
assert_delta('Case 11', prev_pairs, cur_pairs, [('110000000', '118220032')], [])

seed_rows = spark.table(f"{WORK_DB}.output").filter(F.col('REL_ID_REGIE_KLANT') == F.lit('110000000')).count()
assert seed_rows == 0, f"Case 11 seed eligibility mismatch: found {seed_rows} rows with REL_ID_REGIE_KLANT=110000000"
assert_output_has_no_duplicates('Case 11')

Case 11 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,110000000,2026-06-10 10:00:00,2026-06-10 10:05:00,1-1N09820A,Y
4,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
5,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
6,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
7,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
8,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
9,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
107,118492640,2024-02-16 23:34:22,2024-04-24 16:33:50,1-1TWAZ5GR,N
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118220032,110000000
2,118492640,11410074
3,118220032,115379534
4,118492640,117786942
5,118220032,118220032
6,118492640,118492640


Case 11: PASS
Case 11: output invariants OK (unique + N->1 + active seed eligibility)


## Case 12: same rel_id across sources bridges two id_key domains

Scenario:
- Insert `direct_bank` row `(130000200, 1-1N09820A)` with `drc_bnk_f='N'` to anchor on an existing seeded network.
- Insert `ggm_np` row `(130000200, 1-BRIDGE-IKB-12)` using the same rel_id.
- Insert `ggm_np` row `(130000201, 1-BRIDGE-IKB-12)` to validate propagation through the bridged key.

Expected delta/output:
- inserted: `('130000200', '118220032')`, `('130000201', '118220032')`
- removed: none

Additional guard:
- in `edge_snapshot`, rel_id `130000200` must contain both id_keys: `1-1N09820A` and `1-BRIDGE-IKB-12`

In [16]:
prev_pairs = prepare_case_baseline()

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (130000200, to_timestamp('2026-06-10 11:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-1N09820A', 'N')
    """
)

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (130000200, to_timestamp('2026-06-10 11:01:00'), to_timestamp('9999-12-31 00:00:00'), '1-BRIDGE-IKB-12', 'N')
    """
)

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (130000201, to_timestamp('2026-06-10 11:02:00'), to_timestamp('9999-12-31 00:00:00'), '1-BRIDGE-IKB-12', 'N')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 12')
assert_delta('Case 12', prev_pairs, cur_pairs, [('130000200', '118220032'), ('130000201', '118220032')], [])

edge_keys = {
    r['id_key']
    for r in spark.table(f"{STATE_DB}.edge_snapshot")
    .filter(F.col('rel_id') == F.lit('130000200'))
    .select('id_key')
    .collect()
}
assert edge_keys == {'1-1N09820A', '1-BRIDGE-IKB-12'}, f"Case 12 edge bridge mismatch: {sorted(edge_keys)}"
assert_output_has_no_duplicates('Case 12')

Case 12 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y
9,130000200,2026-06-10 11:00:00,9999-12-31 00:00:00,1-1N09820A,N


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N
111,118492640,2025-10-01 07:27:12.034435,9999-12-31 00:00:00,1-1TWAZ5GR,N
112,130000200,2026-06-10 11:01:00,9999-12-31 00:00:00,1-BRIDGE-IKB-12,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640
6,118220032,130000200
7,118220032,130000201


Case 12: PASS
Case 12: output invariants OK (unique + N->1 + active seed eligibility)


## Case 13: multi-seed conflict resolved by latest_then_numeric

Scenario:
- Add two active seeds in `direct_bank`: `130000900` and `130000901` (later `edl_valid_from_dts`).
- Connect both seeds plus `130000902` through shared `ggm_np.ikb_no='1-MULTI-CONNECT'`.

Expected delta/output (default tie-break):
- inserted: `('130000900', '130000901')`, `('130000901', '130000901')`, `('130000902', '130000901')`
- removed: none

Additional guard:
- immediate rerun is idempotent (no inserted/removed pairs).

In [ ]:
prev_pairs = prepare_case_baseline()

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (130000900, to_timestamp('2026-06-12 09:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-MULTI-SEED-A', 'Y')
    """
)

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (130000901, to_timestamp('2026-06-12 10:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-MULTI-SEED-B', 'Y')
    """
)

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (130000900, to_timestamp('2026-06-12 10:10:00'), to_timestamp('9999-12-31 00:00:00'), '1-MULTI-CONNECT', 'N')
    """
)

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (130000901, to_timestamp('2026-06-12 10:11:00'), to_timestamp('9999-12-31 00:00:00'), '1-MULTI-CONNECT', 'N')
    """
)

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (130000902, to_timestamp('2026-06-12 10:12:00'), to_timestamp('9999-12-31 00:00:00'), '1-MULTI-CONNECT', 'N')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 13')
assert_delta(
    'Case 13',
    prev_pairs,
    cur_pairs,
    [('130000900', '130000901'), ('130000901', '130000901'), ('130000902', '130000901')],
    [],
)
winner_rows = (
    spark.table(f"{WORK_DB}.output")
    .filter(F.col('REL_ID') == F.lit('130000902'))
    .select(F.col('REL_ID_REGIE_KLANT').cast('string').alias('REL_ID_REGIE_KLANT'))
    .collect()
)
assert winner_rows and winner_rows[0]['REL_ID_REGIE_KLANT'] == '130000901'
assert_output_has_no_duplicates('Case 13')

prev_pairs_2 = cur_pairs
_ = run_once()
cur_pairs_2 = pair_set(spark.table(f"{WORK_DB}.output"))
assert_delta('Case 13 idempotency', prev_pairs_2, cur_pairs_2, [], [])
assert_output_has_no_duplicates('Case 13 idempotency')

In [ ]:
# Compatibility shim: allow later cases to pass optional run_incremental_update kwargs.
def run_once(**kwargs):
    return run_incremental_update(
        spark,
        source_db=WORK_DB,
        target_db=WORK_DB,
        state_db=STATE_DB,
        show_samples=False,
        **kwargs,
    )

## Case 14: multi-seed conflict resolved by numeric_then_lex

Scenario:
- Reuse multi-seed connectivity, but run with tie-breaker `numeric_then_lex`.
- Two active seeds `130000910` and `130000911` both reach `130000912` through `ggm_np.ikb_no='1-MULTI-CONNECT-2'`.

Expected delta/output:
- inserted: `('130000910', '130000910')`, `('130000911', '130000910')`, `('130000912', '130000910')`
- removed: none

Additional guard:
- immediate rerun is idempotent (no inserted/removed pairs).

In [ ]:
prev_pairs = prepare_case_baseline()

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (130000910, to_timestamp('2026-06-12 09:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-MULTI-SEED-C', 'Y')
    """
)

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (130000911, to_timestamp('2026-06-12 10:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-MULTI-SEED-D', 'Y')
    """
)

for rel_id in (130000910, 130000911, 130000912):
    spark.sql(
        f"""
        INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
        VALUES ({rel_id}, to_timestamp('2026-06-12 10:30:00'), to_timestamp('9999-12-31 00:00:00'), '1-MULTI-CONNECT-2', 'N')
        """
    )

_ = run_once(seed_tie_breaker='numeric_then_lex')
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 14')
assert_delta(
    'Case 14',
    prev_pairs,
    cur_pairs,
    [('130000910', '130000910'), ('130000911', '130000910'), ('130000912', '130000910')],
    [],
)
winner_rows = (
    spark.table(f"{WORK_DB}.output")
    .filter(F.col('REL_ID') == F.lit('130000912'))
    .select(F.col('REL_ID_REGIE_KLANT').cast('string').alias('REL_ID_REGIE_KLANT'))
    .collect()
)
assert winner_rows and winner_rows[0]['REL_ID_REGIE_KLANT'] == '130000910'
assert_output_has_no_duplicates('Case 14')

prev_pairs_2 = cur_pairs
_ = run_once(seed_tie_breaker='numeric_then_lex')
cur_pairs_2 = pair_set(spark.table(f"{WORK_DB}.output"))
assert_delta('Case 14 idempotency', prev_pairs_2, cur_pairs_2, [], [])
assert_output_has_no_duplicates('Case 14 idempotency')

## Case 15: duplicate active seed keys on one rel_id is diagnostic-only

Scenario:
- Insert a second active `direct_bank` seed key for `rel_id=118220032` while keeping open-ended validity.
- This intentionally creates multiple active seed keys for the same rel_id.

Expected delta/output:
- inserted: none
- removed: none

Additional guard:
- output invariants remain valid (uniqueness + active-seed eligibility).

In [ ]:
prev_pairs = prepare_case_baseline()

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (118220032, to_timestamp('2026-06-12 12:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-DUP-ACTIVE-15', 'Y')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 15')
assert_delta('Case 15', prev_pairs, cur_pairs, [], [])
assert_output_has_no_duplicates('Case 15')